In [1]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
from tqdm import tqdm

POS_DIR = "./data/8_telop_position"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 72
GRID_H = 128
BIN_SIZE = 0.1
K_SLOTS = 100
MAX_T = 2000
EVAL_PER_CHANNEL = 5
SEED = 42

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")

# ── 데이터 로드 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

channel_set = set()
samples = []

for channel, path in tqdm(json_paths, desc="로드"):
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        continue

    channel_set.add(channel)
    duration = max(inst["end_sec"] for inst in instances)

    inst_list = []
    for inst in instances:
        inst_list.append({
            "text": inst["text"],
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": inst["grid_x"],
            "y": inst["grid_y"],
            "w": inst["grid_w"],
            "h": inst["grid_h"],
        })

    samples.append({
        "channel": channel,
        "instances": inst_list,
        "duration": duration,
    })

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

print(f"✅ 영상: {len(samples):,}개  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")

✅ 임베딩 로드: 1,448,729개  |  dim=1024
📁 파일 수: 66,400개


로드: 100%|██████████| 66400/66400 [00:13<00:00, 4764.93it/s]


✅ 영상: 59,876개  |  채널: 664개
✅ train: 56,556  |  eval: 3,320


In [2]:
# %% 셀 2: Dataset + DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 8
NUM_WORKERS = 32
SLOT_DIM = 128  # 프레임별 슬롯 임베딩 차원 (메모리 절약)


class TelopMaskDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb

        # Qwen 임베딩 → SLOT_DIM 투영 (dataset 레벨에서 미리)
        # 모든 unique 텍스트에 대해 미리 proj하면 느리니, 필요할 때 계산
        self.emb_proj = nn.Linear(EMB_DIM, SLOT_DIM, bias=False)
        # 학습 안 되는 고정 random projection
        nn.init.orthogonal_(self.emb_proj.weight)
        self.emb_proj.requires_grad_(False)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        duration = max(s["duration"], 0.1)
        T = min(int(np.ceil(duration / BIN_SIZE)), MAX_T)

        channel_id = self.channel2id.get(s["channel"], 0)
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)

        instances = s["instances"]
        inst_starts = np.array([inst["start"] for inst in instances])
        inst_ends = np.array([inst["end"] for inst in instances])
        inst_tlens = np.array([len(inst["text"]) for inst in instances])
        inst_x = np.array([inst["x"] for inst in instances])
        inst_y = np.array([inst["y"] for inst in instances])
        inst_w = np.array([inst["w"] for inst in instances])
        inst_h = np.array([inst["h"] for inst in instances])

        # 인스턴스별 Qwen 임베딩 → proj
        with torch.no_grad():
            inst_embs_full = torch.stack([
                self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
            ])  # (N, 1024)
            inst_embs_proj = self.emb_proj(inst_embs_full)  # (N, SLOT_DIM)

        # ── 프레임별 ──
        features = np.zeros((T, K_SLOTS + 2), dtype=np.float32)
        slot_embs = torch.zeros(T, K_SLOTS, SLOT_DIM)
        slot_mask = torch.zeros(T, K_SLOTS, dtype=torch.bool)
        gt_masks = np.zeros((T, GRID_H, GRID_W), dtype=np.float32)

        for t in range(T):
            t_sec = t * BIN_SIZE
            active = (inst_starts <= t_sec) & (inst_ends > t_sec)
            active_idx = np.where(active)[0]

            features[t, 0] = t / max(T - 1, 1)
            n_active = len(active_idx)
            features[t, 1] = n_active / 20.0

            for j, ai in enumerate(active_idx[:K_SLOTS]):
                features[t, 2 + j] = inst_tlens[ai] / 200.0
                slot_embs[t, j] = inst_embs_proj[ai]
                slot_mask[t, j] = True

            for ai in active_idx:
                x0 = int(np.clip(inst_x[ai], 0, GRID_W - 1))
                y0 = int(np.clip(inst_y[ai], 0, GRID_H - 1))
                x1 = int(np.clip(x0 + inst_w[ai], 1, GRID_W))
                y1 = int(np.clip(y0 + inst_h[ai], 1, GRID_H))
                gt_masks[t, y0:y1, x0:x1] = 1.0

        return {
            "channel_id": torch.tensor(channel_id, dtype=torch.long),
            "channel_emb": channel_emb,              # (EMB_DIM,)
            "features": torch.from_numpy(features),   # (T, 102)
            "slot_embs": slot_embs,                    # (T, K, SLOT_DIM)
            "slot_mask": slot_mask,                    # (T, K)
            "gt_masks": torch.from_numpy(gt_masks),    # (T, 128, 72)
            "T": T,
        }


def collate_fn(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)

    channel_ids = torch.zeros(B, dtype=torch.long)
    channel_embs = torch.zeros(B, EMB_DIM)
    features = torch.zeros(B, max_T, K_SLOTS + 2)
    slot_embs = torch.zeros(B, max_T, K_SLOTS, SLOT_DIM)
    slot_mask = torch.zeros(B, max_T, K_SLOTS, dtype=torch.bool)
    gt_masks = torch.zeros(B, max_T, GRID_H, GRID_W)
    time_mask = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["T"]
        channel_ids[i] = b["channel_id"]
        channel_embs[i] = b["channel_emb"]
        features[i, :T] = b["features"]
        slot_embs[i, :T] = b["slot_embs"]
        slot_mask[i, :T] = b["slot_mask"]
        gt_masks[i, :T] = b["gt_masks"]
        time_mask[i, :T] = True

    return {
        "channel_id": channel_ids,
        "channel_emb": channel_embs,
        "features": features,
        "slot_embs": slot_embs,
        "slot_mask": slot_mask,
        "gt_masks": gt_masks,
        "time_mask": time_mask,
    }


train_ds = TelopMaskDataset(train_samples, channel2id, text2emb)
eval_ds = TelopMaskDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

pos_ratio = batch['gt_masks'][batch['time_mask']].mean().item()
POS_WEIGHT = (1 - pos_ratio) / pos_ratio
print(f"  positive 비율: {pos_ratio:.4f}")
print(f"  POS_WEIGHT: {POS_WEIGHT:.1f}")

✅ 배치 확인
  channel_id: torch.Size([8])
  channel_emb: torch.Size([8, 1024])
  features: torch.Size([8, 601, 102])
  slot_embs: torch.Size([8, 601, 100, 128])
  slot_mask: torch.Size([8, 601, 100])
  gt_masks: torch.Size([8, 601, 128, 72])
  time_mask: torch.Size([8, 601])
  positive 비율: 0.1220
  POS_WEIGHT: 7.2


In [ ]:
# %% 셀 3: 모델 정의
D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 6
D_FF = 512
DROPOUT = 0.1
SPATIAL_CH = 16

CH_ID_DIM = 64
CH_TEXT_DIM = 64
SCALAR_DIM = 64
INTRA_DIM = 64   # intra-frame attention 출력 차원
# concat: 64 + 64 + 64 + 64 = 256 = D_MODEL

POS_WEIGHT = 11.8


class IntraFrameAttention(nn.Module):
    """프레임 내 활성 텔롭들 사이 self-attention → pool"""
    def __init__(self, slot_dim=SLOT_DIM, d_out=INTRA_DIM, n_heads=4):
        super().__init__()
        self.proj_in = nn.Linear(slot_dim, d_out)
        self.attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)
        self.pool_query = nn.Parameter(torch.randn(1, 1, d_out))  # learned query for pooling
        self.pool_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

    def forward(self, slot_embs, slot_mask):
        """
        slot_embs: (B*T, K, SLOT_DIM)
        slot_mask: (B*T, K) — True=유효
        Returns:   (B*T, d_out)
        """
        BT, K, _ = slot_embs.shape

        x = self.proj_in(slot_embs)           # (BT, K, d_out)

        # 활성 슬롯 없는 프레임 처리
        any_active = slot_mask.any(dim=1)      # (BT,)

        # self-attention (패딩 마스킹)
        pad_mask = ~slot_mask                  # True=패딩
        x_att, _ = self.attn(x, x, x, key_padding_mask=pad_mask)  # (BT, K, d_out)

        # attention pooling: learned query로 K개를 1개로 합침
        query = self.pool_query.expand(BT, -1, -1)  # (BT, 1, d_out)
        pooled, _ = self.pool_attn(query, x_att, x_att, key_padding_mask=pad_mask)  # (BT, 1, d_out)
        pooled = pooled.squeeze(1)             # (BT, d_out)

        # 활성 텔롭 없는 프레임은 0벡터
        pooled = pooled * any_active.unsqueeze(-1).float()

        return pooled


class TelopMaskModel(nn.Module):
    def __init__(self, n_channels, emb_dim=EMB_DIM, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS_TEMPORAL,
                 d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        # 채널 인코딩
        self.channel_id_emb = nn.Embedding(n_channels, CH_ID_DIM)
        self.channel_text_proj = nn.Linear(emb_dim, CH_TEXT_DIM)

        # scalar features
        self.scalar_enc = nn.Linear(K_SLOTS + 2, SCALAR_DIM)

        # intra-frame attention
        self.intra_attn = IntraFrameAttention(SLOT_DIM, INTRA_DIM)

        # concat → d_model
        concat_dim = CH_ID_DIM + CH_TEXT_DIM + SCALAR_DIM + INTRA_DIM
        self.token_proj = nn.Sequential(
            nn.Linear(concat_dim, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
        )

        # Temporal Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.temporal_transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers
        )

        # Spatial Decoder
        self.to_spatial = nn.Linear(d_model, SPATIAL_CH * 16 * 9)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(SPATIAL_CH, 8, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(8),
            nn.GELU(),
            nn.ConvTranspose2d(8, 4, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(4),
            nn.GELU(),
            nn.ConvTranspose2d(4, 1, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, channel_id, channel_emb, features, slot_embs, slot_mask, time_mask):
        """
        channel_id:  (B,)
        channel_emb: (B, EMB_DIM)
        features:    (B, T, 102)
        slot_embs:   (B, T, K, SLOT_DIM)
        slot_mask:   (B, T, K)
        time_mask:   (B, T)
        Returns:     (B, T, 128, 72)
        """
        B, T, _ = features.shape

        # ① 채널
        ch_id = self.channel_id_emb(channel_id).unsqueeze(1).expand(-1, T, -1)  # (B, T, 64)
        ch_text = self.channel_text_proj(channel_emb).unsqueeze(1).expand(-1, T, -1)  # (B, T, 64)

        # ② scalar
        scalar = self.scalar_enc(features)  # (B, T, 64)

        # ③ intra-frame attention
        slot_embs_flat = slot_embs.view(B * T, K_SLOTS, SLOT_DIM)
        slot_mask_flat = slot_mask.view(B * T, K_SLOTS)
        intra_out = self.intra_attn(slot_embs_flat, slot_mask_flat)  # (B*T, 64)
        intra_out = intra_out.view(B, T, INTRA_DIM)  # (B, T, 64)

        # ④ concat → 투영
        concat = torch.cat([ch_id, ch_text, scalar, intra_out], dim=-1)  # (B, T, 256)
        tokens = self.token_proj(concat)  # (B, T, 256)

        # ⑤ Temporal Transformer
        padding_mask = ~time_mask
        out = self.temporal_transformer(tokens, src_key_padding_mask=padding_mask)  # (B, T, 256)

        # ⑥ Spatial Decoder
        spatial_flat = self.to_spatial(out)
        spatial = spatial_flat.view(B * T, SPATIAL_CH, 16, 9)
        masks = self.decoder(spatial)
        masks = masks.view(B, T, GRID_H, GRID_W)

        return masks


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TelopMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")

🖥️  Device: cuda
📐 파라미터: 3,980,005


In [ ]:
# %% 셀 4: 학습
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
SAVE_DIR = "./model/8_layout_mask_intra_model"
os.makedirs(SAVE_DIR, exist_ok=True)


def compute_loss(pred, gt, time_mask):
    mask_3d = time_mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred)
    pred_valid = pred[mask_3d]
    gt_valid = gt[mask_3d]
    pos_weight = torch.tensor([POS_WEIGHT], device=pred.device)
    return F.binary_cross_entropy_with_logits(pred_valid, gt_valid, pos_weight=pos_weight)


@torch.no_grad()
def compute_metrics(pred, gt, time_mask, threshold=0.5):
    mask_3d = time_mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred)
    pred_bin = (torch.sigmoid(pred[mask_3d]) > threshold).float()
    gt_valid = gt[mask_3d]

    tp = (pred_bin * gt_valid).sum().item()
    fp = (pred_bin * (1 - gt_valid)).sum().item()
    fn = ((1 - pred_bin) * gt_valid).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    return {"precision": precision, "recall": recall, "f1": f1}


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0, 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_id = batch["channel_id"].to(device)
        channel_emb = batch["channel_emb"].to(device)
        features = batch["features"].to(device)
        slot_embs = batch["slot_embs"].to(device)
        slot_mask = batch["slot_mask"].to(device)
        gt_masks = batch["gt_masks"].to(device)
        time_mask = batch["time_mask"].to(device)

        pred = model(channel_id, channel_emb, features, slot_embs, slot_mask, time_mask)
        loss = compute_loss(pred, gt_masks, time_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0, 0
    all_metrics = {"precision": [], "recall": [], "f1": []}

    with torch.no_grad():
        for batch in eval_loader:
            channel_id = batch["channel_id"].to(device)
            channel_emb = batch["channel_emb"].to(device)
            features = batch["features"].to(device)
            slot_embs = batch["slot_embs"].to(device)
            slot_mask = batch["slot_mask"].to(device)
            gt_masks = batch["gt_masks"].to(device)
            time_mask = batch["time_mask"].to(device)

            pred = model(channel_id, channel_emb, features, slot_embs, slot_mask, time_mask)
            loss = compute_loss(pred, gt_masks, time_mask)
            metrics = compute_metrics(pred, gt_masks, time_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1
            for k, v in metrics.items():
                all_metrics[k].append(v)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss = eval_loss_sum / max(eval_batches, 1)
    avg_m = {k: np.mean(v) for k, v in all_metrics.items()}
    lr_now = optimizer.param_groups[0]["lr"]

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m['precision']:.3f}  R={avg_m['recall']:.3f}  F1={avg_m['f1']:.3f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[1/50] train:   2%|▏         | 164/7070 [00:53<16:01,  7.18it/s] 